# Clasificador liviano de recortes (H4)

Segunda etapa del pipeline: cada caja candidata del detector se recorta y un clasificador binario decide si es **fuego o humo real** o un **elemento confundible** (nubes, niebla, reflejos, luces). Reduce falsos positivos **por contenido** y complementa al criterio temporal, que los reduce **por persistencia**.

1. **Dataset de recortes**, construido **solo con train** (si se usan val o test, se filtra información a la evaluación):
   - `real`: cajas de ground truth de train (fuego y humo).
   - `confundible`: falsos positivos del detector sobre las 6.458 imágenes negativas de train.
2. **Entrenamiento** de `yolo11n-cls` preentrenado (misma herramienta que el detector).
3. **Evaluación:**
   - Sobre recortes de validación: accuracy, precision y recall de `confundible`.
   - **Sobre el test del detector** (la medida que importa para H4): P y R por clase y falsos positivos por imagen negativa, con y sin clasificador, y la curva al variar el umbral del clasificador.
4. Una fila en `results/master.csv`.

El recorte (margen de contexto) sale de `recortes.py`, el mismo que usan `inferencia_video.ipynb` y la demo. Funciona en local, Colab o Kaggle; fuera de local hay que subir `recortes.py` y `evaluacion.py` a la sesión.

## 1. Plataforma y dependencias

In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
from datetime import date
from pathlib import Path

# Kaggle se reconoce por su variable de entorno, no por /kaggle/input: Colab también puede tener esa carpeta
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
# Colab: el módulo ya cargado, o las variables de entorno que define el runtime
IN_COLAB = "google.colab" in sys.modules or any(k.startswith("COLAB_") for k in os.environ)
PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"

ULTRALYTICS_VERSION = "8.4.157"
if PLATFORM != "local":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS_VERSION}", "kagglehub"],
        check=True,
    )
DRIVE_OK = False
if PLATFORM == "colab":
    # Drive permite retomar un run en otra sesión. Si no se puede montar (Colab desde VS Code,
    # ventana de autorización bloqueada, varias cuentas de Google abiertas), se sigue en el disco
    # de la sesión: funciona igual, pero hay que bajar el zip del final antes de cerrar.
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_OK = True
    except Exception as e:
        print(f"[!] No se pudo montar Drive ({e}). Se guarda en /content/incendios-tp: "
              "bajar el zip del final antes de cerrar la sesión.")

try:
    import evaluacion
    import recortes
except ImportError as e:
    raise ImportError("Faltan recortes.py / evaluacion.py: subirlos a la sesión (panel Files) o clonar el repo") from e

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from tqdm.auto import tqdm
from ultralytics import YOLO

WORK_DIR = {
    "kaggle": Path("/kaggle/working"),
    "colab": Path("/content/drive/MyDrive/incendios-tp" if DRIVE_OK else "/content/incendios-tp"),
    "local": Path.cwd(),
}[PLATFORM]
RUNS_DIR = WORK_DIR / "runs"
RESULTS_DIR = WORK_DIR / "results"
FIG_DIR = WORK_DIR / "figures"
# Los recortes van a disco local (en Colab, fuera de Drive: mucho más rápido); se regeneran si faltan
CLF_DATA = {"kaggle": Path("/tmp/clasificador"), "colab": Path("/content/clasificador"),
            "local": WORK_DIR / "data" / "clasificador"}[PLATFORM]
for d in (RUNS_DIR, RESULTS_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
DEVICE_NAME = torch.cuda.get_device_name(0) if DEVICE == 0 else "cpu"
print(f"Plataforma: {PLATFORM} | ultralytics {ultralytics.__version__} | device: {DEVICE_NAME}")

In [ ]:
DATASET_SLUG = "sayedgamal99/smoke-fire-detection-yolo"
SPLITS = ("train", "val", "test")
CLASS_NAMES = evaluacion.CLASS_NAMES

def find_split_root(base: Path) -> Path | None:
    for train_dir in base.rglob("train"):
        root = train_dir.parent
        if all((root / s / sub).is_dir() for s in SPLITS for sub in ("images", "labels")):
            return root
    return None

if PLATFORM == "kaggle":
    SPLIT_ROOT = find_split_root(Path("/kaggle/input"))
elif PLATFORM == "colab":
    import kagglehub
    SPLIT_ROOT = find_split_root(Path(kagglehub.dataset_download(DATASET_SLUG)))
else:
    SPLIT_ROOT = find_split_root(Path.cwd() / "data")
assert SPLIT_ROOT is not None, "No se encontró D-Fire"

leer_gt = evaluacion.leer_gt

def imagenes(split: str) -> list[tuple[Path, Path]]:
    imgs = sorted((SPLIT_ROOT / split / "images").iterdir())
    return [(p, SPLIT_ROOT / split / "labels" / f"{p.stem}.txt") for p in imgs]

print("Dataset:", SPLIT_ROOT)

## 2. Configuración

**Qué detector produce los falsos positivos (`MINE_RUN`).** Se recomienda **R5** (`_noneg`): no vio las imágenes negativas al entrenar, así que sus errores sobre ellas se parecen a los que el sistema va a cometer en escenas nuevas. Con R1, que sí las vio, salen muy pocos falsos positivos (las memorizó) y la clase `confundible` queda chica y poco representativa. Si R5 no está, usar R1 con `MINE_CONF` bajo.

**Con qué detector se evalúa el sistema (`EVAL_RUN`)**: el detector que se va a desplegar (el mejor según `master.csv`).

Para minar se usa confianza baja (`MINE_CONF = 0.05`, como en inferencia) y como máximo `MAX_FP_POR_IMAGEN` cajas por imagen, para que unas pocas imágenes con muchos falsos positivos no dominen la clase. Las cajas con lado menor a `MIN_LADO_PX` se descartan: con tan pocos píxeles el recorte no aporta información.

`RATIO_REAL`: cantidad de recortes `real` por cada `confundible` (se submuestrea `real`, que es mucho más grande). Ultralytics no pondera clases en clasificación, así que balancear evita que el clasificador aprenda a decir siempre `real`.

In [ ]:
SEED = 42
MINE_RUN = "yolo11n_640_noneg_s42"   # detector para minar falsos positivos (R5 recomendado)
EVAL_RUN = "yolo11n_640_neg_s42"     # detector del sistema, para la evaluación sobre test
DET_IMGSZ = 640

MINE_CONF = 0.05
MAX_FP_POR_IMAGEN = 5
MIN_LADO_PX = 8
RATIO_REAL = 1.0
VAL_FRAC = 0.15
REBUILD = False                       # True: rehacer los recortes aunque ya existan

CLF_NAME = "clasificador"             # inferencia_video.ipynb espera runs/clasificador/weights/best.pt
CLF_IMGSZ = 224
EPOCHS = 30
BATCH = 64

TAU_EVAL = 0.40                       # punto de operación del detector para comparar con / sin clasificador
UMBRAL_CLF = 0.50                     # p_real mínima para confirmar
CLF_CONF_MIN = 0.25                   # igual que en inferencia_video.ipynb

SMOKE_TEST = False                    # True: pocas imágenes y 1 época, para probar el pipeline sin GPU

def pesos(run: str):
    p = RUNS_DIR / run / "weights" / "best.pt"
    if p.exists():
        return p
    if SMOKE_TEST:
        print(f"[!] No está {run}: SMOKE_TEST con yolo11n.pt de COCO")
        return "yolo11n.pt"
    raise FileNotFoundError(f"No está {p}: entrenar ese run con train.ipynb (o cambiar MINE_RUN / EVAL_RUN)")

N_MAX = 150 if SMOKE_TEST else None   # imágenes por grupo en SMOKE_TEST
if SMOKE_TEST:
    EPOCHS, BATCH = 1, 16
random.seed(SEED)
np.random.seed(SEED)

## 3. Dataset de recortes

Se arma un *pool* con todos los recortes de cada clase (anotando de qué imagen salen) y después se separa train / val **por imagen**: todos los recortes de una misma imagen van al mismo lado. Igual conviene leer la validación como optimista: D-Fire tiene frames casi idénticos de una misma cámara (ver `eda.ipynb`). La medida honesta es la de la sección 6, sobre el test del detector.

Estructura final (formato de clasificación de Ultralytics): `CLF_DATA/{train,val}/{real,confundible}/*.jpg`. Si ya existe con los mismos parámetros, no se rehace.

In [ ]:
def params_dataset() -> dict:
    return {"mine_run": MINE_RUN, "mine_conf": MINE_CONF, "max_fp": MAX_FP_POR_IMAGEN,
            "min_lado": MIN_LADO_PX, "ratio_real": RATIO_REAL, "val_frac": VAL_FRAC,
            "margen": recortes.MARGEN, "seed": SEED, "smoke_test": SMOKE_TEST}

meta_path = CLF_DATA / "meta.json"
dataset_ok = (not REBUILD and meta_path.exists()
              and json.loads(meta_path.read_text(encoding="utf-8"))["params"] == params_dataset())

if dataset_ok:
    print("Dataset de recortes ya existe con estos parámetros:", CLF_DATA)
else:
    shutil.rmtree(CLF_DATA, ignore_errors=True)
    pool_dir = CLF_DATA / "_pool"
    for c in (recortes.CLASE_REAL, "confundible"):
        (pool_dir / c).mkdir(parents=True)

    train = imagenes("train")
    con_obj = [(i, l) for i, l in train if l.read_text().strip()]
    negativas = [(i, l) for i, l in train if not l.read_text().strip()]
    if N_MAX:
        con_obj, negativas = random.sample(con_obj, N_MAX), random.sample(negativas, N_MAX)

    pool = []

    # real: cajas de ground truth
    for img_path, lbl_path in tqdm(con_obj, desc="real (GT)"):
        img = cv2.imread(str(img_path))
        H, W = img.shape[:2]
        for k, g in enumerate(leer_gt(lbl_path, W, H)):
            if min(g["x2"] - g["x1"], g["y2"] - g["y1"]) < MIN_LADO_PX:
                continue
            dst = pool_dir / recortes.CLASE_REAL / f"{img_path.stem}_{k}.jpg"
            cv2.imwrite(str(dst), recortes.recortar(img, g["x1"], g["y1"], g["x2"], g["y2"]))
            pool.append({"imagen": img_path.stem, "clase": recortes.CLASE_REAL, "origen": g["clase"], "conf": np.nan, "ruta": dst})

    # confundible: falsos positivos del detector sobre negativas (cualquier caja es un falso positivo)
    minero = YOLO(pesos(MINE_RUN))
    for img_path, _ in tqdm(negativas, desc="confundible (FP)"):
        r = minero.predict(str(img_path), imgsz=DET_IMGSZ, conf=MINE_CONF, device=DEVICE, verbose=False)[0]
        orden = np.argsort(-r.boxes.conf.cpu().numpy())[:MAX_FP_POR_IMAGEN]
        for k in orden:
            x1, y1, x2, y2 = r.boxes.xyxy[k].tolist()
            if min(x2 - x1, y2 - y1) < MIN_LADO_PX:
                continue
            dst = pool_dir / "confundible" / f"{img_path.stem}_{k}.jpg"
            cv2.imwrite(str(dst), recortes.recortar(r.orig_img, x1, y1, x2, y2))
            pool.append({"imagen": img_path.stem, "clase": "confundible", "origen": r.names[int(r.boxes.cls[k])],
                         "conf": float(r.boxes.conf[k]), "ruta": dst})

    pool = pd.DataFrame(pool)
    print(pool.groupby(["clase", "origen"]).size().rename("recortes").to_frame())

In [ ]:
if not dataset_ok:
    # Split por imagen
    imgs = sorted(pool["imagen"].unique())
    rng = np.random.default_rng(SEED)
    val_imgs = set(rng.choice(imgs, size=int(len(imgs) * VAL_FRAC), replace=False))
    pool["split"] = np.where(pool["imagen"].isin(val_imgs), "val", "train")

    # Balanceo: submuestrear real a RATIO_REAL × confundible, por split
    partes = []
    for split, g in pool.groupby("split"):
        conf_g = g[g["clase"] == "confundible"]
        real_g = g[g["clase"] == recortes.CLASE_REAL]
        n_real = min(len(real_g), int(round(RATIO_REAL * len(conf_g))))
        partes += [conf_g, real_g.sample(n=n_real, random_state=SEED)]
    final = pd.concat(partes)

    for r in final.itertuples():
        dst = CLF_DATA / r.split / r.clase / r.ruta.name
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(r.ruta, dst)
    shutil.rmtree(CLF_DATA / "_pool")

    conteo = final.groupby(["split", "clase"]).size().unstack(fill_value=0)
    meta_path.write_text(json.dumps({
        "params": params_dataset(),
        "conteo": {s: row.to_dict() for s, row in conteo.iterrows()},
        "pool": pool.groupby("clase").size().to_dict(),
    }, indent=2, default=int), encoding="utf-8")

meta = json.loads(meta_path.read_text(encoding="utf-8"))
print("Recortes disponibles antes de balancear:", meta["pool"])
pd.DataFrame(meta["conteo"]).T

In [ ]:
# Ejemplos de cada clase (lo que ve el clasificador)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax_row, clase in zip(axes, (recortes.CLASE_REAL, "confundible")):
    rutas = sorted((CLF_DATA / "train" / clase).glob("*.jpg"))
    for ax, p in zip(ax_row, random.Random(SEED).sample(rutas, min(8, len(rutas)))):
        ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
        ax.set_title(p.stem, fontsize=7)
    for ax in ax_row:
        ax.axis("off")
    ax_row[0].text(-0.15, 0.5, clase, transform=ax_row[0].transAxes, rotation=90, va="center", ha="right")
fig.suptitle("Recortes de entrenamiento del clasificador")
fig.tight_layout()
fig.savefig(FIG_DIR / "clasificador_ejemplos.png", dpi=150, bbox_inches="tight")

## 4. Entrenamiento

`yolo11n-cls` preentrenado en ImageNet, con los aumentos por defecto de Ultralytics para clasificación. Sin early stopping (`patience = EPOCHS`), como el detector. Si la sesión se corta, retoma desde `last.pt`.

In [ ]:
CLF_DIR = RUNS_DIR / CLF_NAME
last_pt, best_pt = CLF_DIR / "weights" / "last.pt", CLF_DIR / "weights" / "best.pt"

def epochs_done(run_dir: Path) -> int:
    csv = run_dir / "results.csv"
    return len(pd.read_csv(csv)) if csv.exists() else 0

if epochs_done(CLF_DIR) >= EPOCHS and best_pt.exists():
    print(f"{CLF_NAME} ya está entrenado ({epochs_done(CLF_DIR)} épocas), se saltea.")
elif last_pt.exists():
    YOLO(last_pt).train(resume=True)
else:
    YOLO("yolo11n-cls.pt").train(
        data=str(CLF_DATA), epochs=EPOCHS, patience=EPOCHS, imgsz=CLF_IMGSZ, batch=BATCH,
        seed=SEED, deterministic=True, device=DEVICE,
        project=str(RUNS_DIR), name=CLF_NAME, exist_ok=True, plots=True,
    )

## 5. Evaluación sobre recortes de validación

Precision y recall de `confundible`: qué tan bien separa los falsos positivos de las detecciones reales cuando ya están recortados.

In [ ]:
clf = YOLO(best_pt)
m = clf.val(data=str(CLF_DATA), split="val", imgsz=CLF_IMGSZ, batch=BATCH, device=DEVICE,
            project=str(CLF_DIR), name="eval_val", exist_ok=True, plots=True, verbose=False)

cm = m.confusion_matrix.matrix  # [predicha, verdadera]
names = clf.names
i_conf = [i for i, n in names.items() if n == "confundible"][0]
tp = cm[i_conf, i_conf]
clf_val = {
    "clf_val_top1": float(m.top1),
    "clf_val_P_confundible": float(tp / cm[i_conf, :].sum()) if cm[i_conf, :].sum() else np.nan,
    "clf_val_R_confundible": float(tp / cm[:, i_conf].sum()) if cm[:, i_conf].sum() else np.nan,
}
print(pd.DataFrame(cm[:len(names), :len(names)].astype(int),
                   index=[f"pred_{n}" for n in names.values()], columns=[f"true_{n}" for n in names.values()]))
clf_val

## 6. Efecto sobre el detector en test (H4)

El detector del sistema (`EVAL_RUN`) corre sobre las 4.306 imágenes de test con `conf > CLF_CONF_MIN`; cada caja se recorta y el clasificador le asigna `p_real`. Después se compara, al umbral operativo `TAU_EVAL`:

- **Sin clasificador:** toda caja con `conf > τ` es una detección.
- **Con clasificador:** además se exige `p_real ≥ UMBRAL_CLF`.

Métricas, por clase: TP, FP, FN, precision y recall (`evaluacion.py`: match con IoU ≥ 0,5, de mayor a menor confianza, una caja de GT por predicción). Además, **falsos positivos por imagen negativa** (las 2.005 de test sin fuego ni humo): lo que el clasificador tiene que bajar sin perder mucho recall.

Las predicciones se guardan en `results/clasificador_test_preds.csv` para no recalcularlas.

In [ ]:
preds_path = RESULTS_DIR / f"{CLF_NAME}_test_preds.csv"
gts_path = RESULTS_DIR / f"{CLF_NAME}_test_gts.csv"
idx_real = recortes.indice_clase_real(clf.names)

if preds_path.exists() and gts_path.exists() and not SMOKE_TEST:
    preds, gts = pd.read_csv(preds_path), pd.read_csv(gts_path)
    print("Predicciones de test ya calculadas:", preds_path)
else:
    detector = YOLO(pesos(EVAL_RUN))
    test = imagenes("test")
    if N_MAX:
        test = random.Random(SEED).sample(test, N_MAX)
    pred_rows, gt_rows = [], []
    for img_path, lbl_path in tqdm(test, desc="test"):
        r = detector.predict(str(img_path), imgsz=DET_IMGSZ, conf=CLF_CONF_MIN, device=DEVICE, verbose=False)[0]
        H, W = r.orig_shape
        gt_rows += [{"imagen": img_path.stem, **g} for g in leer_gt(lbl_path, W, H)]
        if not len(r.boxes):
            continue
        xyxy = r.boxes.xyxy.tolist()
        crops = [recortes.recortar(r.orig_img, *b) for b in xyxy]
        probs = clf.predict(crops, imgsz=CLF_IMGSZ, device=DEVICE, verbose=False)
        for (x1, y1, x2, y2), conf, cls, pr in zip(xyxy, r.boxes.conf.tolist(), r.boxes.cls.tolist(), probs):
            pred_rows.append({"imagen": img_path.stem, "clase": r.names[int(cls)], "conf": conf,
                              "x1": x1, "y1": y1, "x2": x2, "y2": y2, "p_real": float(pr.probs.data[idx_real])})
    preds = pd.DataFrame(pred_rows, columns=["imagen", "clase", "conf", "x1", "y1", "x2", "y2", "p_real"])
    gts = pd.DataFrame(gt_rows, columns=["imagen", "clase", "x1", "y1", "x2", "y2", "ancho_img", "alto_img"])
    all_imgs = pd.DataFrame({"imagen": [p.stem for p, _ in test]})
    all_imgs.to_csv(RESULTS_DIR / f"{CLF_NAME}_test_imagenes.csv", index=False)
    preds.to_csv(preds_path, index=False)
    gts.to_csv(gts_path, index=False)

test_imgs = pd.read_csv(RESULTS_DIR / f"{CLF_NAME}_test_imagenes.csv")["imagen"]
negativas_test = sorted(set(test_imgs) - set(gts["imagen"]))
print(f"{len(test_imgs)} imágenes de test ({len(negativas_test)} negativas) | {len(preds)} predicciones | {len(gts)} cajas GT")

In [ ]:
def evaluar(tau: float, umbral_clf: float | None = None) -> dict:
    return evaluacion.evaluar(preds, gts, negativas_test, tau, umbral_clf)

comparacion = pd.DataFrame([evaluar(TAU_EVAL, None), evaluar(TAU_EVAL, UMBRAL_CLF)],
                           index=["sin clasificador", f"con clasificador (p_real ≥ {UMBRAL_CLF})"])
comparacion.T

In [ ]:
# Barrido del umbral del clasificador al τ operativo: recall por clase vs. falsos positivos por imagen negativa
umbrales = [None, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
curva = pd.DataFrame([evaluar(TAU_EVAL, u) for u in umbrales])
curva.to_csv(RESULTS_DIR / f"{CLF_NAME}_curva_umbral.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5))
for clase, color in (("fire", "tab:red"), ("smoke", "tab:gray")):
    ax.plot(curva["FP_por_img_neg"], curva[f"R_{clase}"], "o-", color=color, label=clase)
    sin = curva.iloc[0]
    ax.plot(sin["FP_por_img_neg"], sin[f"R_{clase}"], "s", ms=10, mfc="none", color=color)
for _, r in curva.iloc[1::2].iterrows():
    ax.annotate(f"{r['umbral_clf']:.1f}", (r["FP_por_img_neg"], r["R_fire"]), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("Falsos positivos por imagen negativa (test)")
ax.set_ylabel("Recall")
ax.set_title(f"Efecto del clasificador de recortes (τ = {TAU_EVAL}); □ = sin clasificador")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIG_DIR / "clasificador_recall_vs_fp.png", dpi=150, bbox_inches="tight")
fig.savefig(FIG_DIR / "clasificador_recall_vs_fp.pdf", bbox_inches="tight")
curva

## 7. `results/master.csv`

Una fila para el clasificador (`run_id = CLF`), con las métricas sobre recortes de validación y la comparación sin / con clasificador en test. Igual que en `train.ipynb`: fuera de local se baja el zip y se combina con `merge_master`.

In [ ]:
sin, con = comparacion.iloc[0], comparacion.iloc[1]
row = {
    "run": f"{CLF_NAME}_{MINE_RUN}" + ("_smoke" if SMOKE_TEST else ""), "run_id": "CLF", "modelo": "yolo11n-cls",
    "imgsz": CLF_IMGSZ, "train_set": f"recortes ({MINE_RUN})", "seed": SEED, "epochs": epochs_done(CLF_DIR),
    "batch": BATCH, "plataforma": PLATFORM, "device": DEVICE_NAME, "ultralytics": ultralytics.__version__,
    "fecha": date.today().isoformat(), "eval_run": EVAL_RUN, "tau_eval": TAU_EVAL, "umbral_clf": UMBRAL_CLF,
    **{k: round(v, 4) for k, v in clf_val.items()},
}
for nombre, fila in (("sin_clf", sin), ("con_clf", con)):
    for k in ("P_smoke", "R_smoke", "P_fire", "R_fire", "FP_por_img_neg", "pct_img_neg_con_FP"):
        row[f"test_{k}_{nombre}"] = round(float(fila[k]), 4)

MASTER = RESULTS_DIR / ("master_smoke.csv" if SMOKE_TEST else "master.csv")
master = pd.read_csv(MASTER) if MASTER.exists() else pd.DataFrame()
if not master.empty:
    master = master[master["run"] != row["run"]]
master = pd.concat([master, pd.DataFrame([row])], ignore_index=True)
master.to_csv(MASTER, index=False)
print("Actualizado:", MASTER)
pd.Series(row).to_frame(row["run"])

In [ ]:
import zipfile

# Fuera de local: zip con runs/clasificador (con best.pt), CSV de test y la fila parcial para merge_master
if PLATFORM != "local":
    archive = WORK_DIR / f"{CLF_NAME}_resultados.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for p in CLF_DIR.rglob("*"):
            if p.is_file() and p.name != "last.pt":
                z.write(p, Path("runs") / p.relative_to(RUNS_DIR))
        for p in RESULTS_DIR.glob(f"{CLF_NAME}_*.csv"):
            z.write(p, Path("results") / p.name)
        for p in FIG_DIR.glob("clasificador_*"):
            z.write(p, Path("figures") / p.name)
        z.writestr(f"results/master_{row['run']}.csv", pd.DataFrame([row]).to_csv(index=False))
    print("Zip:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")